# Топологические и статистические метрики схожести распределений

Курсовой проект. Переработанная и дополненная версия `topology_metrics_RnD.ipynb`.

**Цель** — посмотреть, как разные метрики схожести двух облаков точек (данные $P$ vs модель $Q$) реагируют на контролируемую деформацию распределений, и проверить гипотезу научного руководителя: *precision/recall-метрики сильнее коррелируют между собой, чем с симметричными метриками-расходимостями, и наоборот*.

## Метрики

| Метрика | Тип | Что измеряет |
|---|---|---|
| **MTD** (Manifold Topology Divergence) | асимметричная, топологическая | сумму длин $H_1$-баров кросс-баркода пары облаков |
| **RTD** (Representation Topology Divergence) | симметричная, топологическая | расхождение топологий внутренних структур двух облаков |
| **Improved precision / recall** | асимметричная | долю точек одного облака внутри kNN-многообразия другого |
| **MMD** (Maximum Mean Discrepancy) | симметричная | расстояние между распределениями в RKHS (RBF-ядро) |
| **Fréchet distance** | симметричная | $W_2$-расстояние между гауссианами, подогнанными к облакам (формула FID) |
| **JS divergence** | симметричная | дивергенцию Дженсена–Шеннона между плотностями (kNN-оценка) |

## Структура ноутбука

1. **Генерация данных** — сэмплеры для управляемых 2D-экспериментов
2. **Метрики** — класс-обертка `TopologyMetrics`, детали реализации библиотек, sanity-проверки
3. **Синтетические эксперименты** — кольцо/диск/смесь с деформацией (нумерация из переписки)
4. **Корреляционный анализ** — проверка гипотезы
5. **Реальные данные** — эмбеддинги CLIP: MNIST, Fashion-MNIST, CIFAR-10-C
6. **Итоги и литература**

> **Требования к окружению:** GPU-runtime Google Colab — ripser++ (используется в MTD и RTD) собирается только с CUDA. Improved precision/recall и статистические метрики (MMD, Fréchet, JS) работают и на CPU: TF запускается с `allow_soft_placement`.


## Задачи из переписки и их статус

| # | Задача (из переписки) | Было в RnD-ноутбуке | Где сейчас |
|---|---|---|---|
| 1 | 100 точек: гауссиана + равномерный квадрат; MTD в обе стороны, precision, recall | не те распределения (две одинаковые усеченные гауссианы) | § 2.3 |
| 2 | FID | не сделано | § 2.1: `frechet_distance` — формула FID на сырых фичах; на эмбеддингах — § 5 |
| 3 | Симметричные метрики: JS-дивергенция для наборов векторов | не сделано | § 2.1: `js_divergence` (kNN-оценка энтропий) |
| 4 | MMD | сделано, фиксированное `gamma=1.0` | § 2.1: несмещенная оценка + медианная эвристика |
| 5 | Класс-обертка: два облака на вход, метрики через методы | сделано | § 2.1: `TopologyMetrics` |
| 6 | Форк improved-precision-recall: setup.py, запуск в Colab | сделано | § 0 (установка) |
| 7 | Разобраться с деталями реализации rtd / mtd / precision_recall | вопросы в комментариях ячеек | § 2.2 (ответы на все вопросы) |
| 8 | MTD: по каким гомологиям считается расстояние (H0 / H1) | не сделано | § 2.1: `mtd_homology`; § 2.2 |
| 9 | Разные сиды для двух облаков вместо копий | частично (`deepcopy`) | § 3 (везде разные сиды) |
| 10 | Эксперимент 1: два кольца, одно двигаем | сделано | § 3.1 |
| 11 | Эксперименты 2а/2б: диск (данные) vs кольцо (модель) и наоборот | только 2а | § 3.2–3.3 |
| 12 | Эксперименты 3а/3б: гауссиана vs раздвигающаяся смесь и наоборот | только 3а | § 3.4–3.5 |
| 13 | Смесь: параметр числа компонент (4, 8, ...) | сделано | § 3.6 |
| 14 | Корреляция precision/recall-метрик между собой и с симметричными | не сделано | § 4 |
| 15 | Реальные данные через предобученную модель (эмбеддинги) | обсуждалось | § 5 (CLIP) |

**Исправленные ошибки старого ноутбука:**

- опечатка `figsize=(10,5b))` → `SyntaxError` в разделе Sphere metrics;
- лишний сдвиг `moved_points[:, 0] += step` в эксперименте со смесью (сдвигал облако, хотя эксперимент про раздвигание компонент);
- избыточный двойной вызов `knn_precision_recall_features` со swapped аргументами (второй вызов возвращает те же precision/recall с точностью до перестановки);
- MMD с фиксированным `gamma=1.0` — метрика зависела от масштаба данных (теперь медианная эвристика);
- загадка «облака совпадают полностью, а MTD не ноль» — облака совпадали только визуально: это были две выборки одного закона с разными сидами; ненулевой MTD и precision/recall < 1 — корректный шум сэмплирования, а не баг (разбор в § 2.3);
- шестикратное копирование кода экспериментов → общий пайплайн `run_experiment` (§ 3).


## 0. Установка

Установка занимает ~5 минут (ripser++ компилируется с CUDA). Повторный запуск ячеек безопасен: `git clone` пропустится, `pip install` переустановит то же самое.

Ключевые зависимости:

- **ripser++** — GPU-вычисление персистентных гомологий (нужен MTD и RTD);
- **MTopDiv** — Manifold Topology Divergence;
- **RTD** — Representation Topology Divergence;
- **improved-precision-and-recall-metric** — форк [AlimAlb/improved-precision-and-recall-metric](https://github.com/AlimAlb/improved-precision-and-recall-metric): добавлен `setup.py` и включен режим совместимости с TF1, чтобы ставился в Colab без костылей (задача № 6 из таблицы выше).


In [ ]:
REPO_URL = 'https://github.com/AlimAlb/tda_metrics_project.git'

!git clone $REPO_URL tda_metrics_repo 2> /dev/null || echo 'репозиторий: уже склонирован'
!pip install -q ./tda_metrics_repo

# ripser++: требуется GPU-runtime (сборка с CUDA)
!git clone --recursive https://github.com/simonzhang00/ripser-plusplus.git 2> /dev/null || echo 'ripser-plusplus: уже склонирован'
!pip install -q ./ripser-plusplus

# MTD (Manifold Topology Divergence)
!git clone https://github.com/IlyaTrofimov/MTopDiv.git 2> /dev/null || echo 'MTopDiv: уже склонирован'
!pip install -q ./MTopDiv

# RTD (Representation Topology Divergence)
!pip install -q git+https://github.com/IlyaTrofimov/RTD.git

# improved precision/recall: форк с setup.py и правками TF1-режима
!git clone https://github.com/AlimAlb/improved-precision-and-recall-metric.git 2> /dev/null || echo 'improved-precision-and-recall: уже склонирован'
!pip install -q ./improved-precision-and-recall-metric


### Импорты

`precision_recall` написан под TF1, поэтому перед импортом включаем режим совместимости TensorFlow (`disable_eager_execution` + `disable_v2_behavior`). Импорт `mtd` и `rtd` сразу проверяет, что ripser++ собрался: если сборка не удалась — упадет здесь с понятной ошибкой.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from tda_metrics import (
    TopologyMetrics, np_random_seed,
    sample_gaussian, sample_uniform_cube, sample_ball, sample_ring, sample_gaussian_mixture,
    constant_cloud, translated_cloud, run_experiment, show_steps,
    plot_clouds, plot_trajectories,
    trajectories_correlation, plot_correlation_heatmap, group_correlation_summary,
)

print('tda_metrics: импортирован успешно')


## 1. Генерация данных

Все сэмплеры детерминированы через `seed` (`np.random.default_rng`) и возвращают массив `(n, dim)`.

### Равномерный диск (шар)

Чтобы точки заполняли диск **равномерно по площади**, нельзя брать радиус как $U[0, R]$ — точки сгустятся у центра. Нужно, чтобы $P(r' < r)$ равнялась доле объема:

$$P(r' < r) = \frac{V(r)}{V(R)} = \left(\frac{r}{R}\right)^{dim}$$

Пусть $U \sim U[0,1]$, тогда $U = (r/R)^{dim} \Rightarrow r = R \cdot U^{1/dim}$.

### Толстое кольцо

Направление — нормализованный гауссовский вектор (равномерен на сфере), радиус — $N(R,\ \text{thickness}/4)$: «правило $4\sigma$», т.е. эффективная толщина кольца $\approx$ `thickness` ($\pm 2\sigma$). При `thickness=0` — окружность.

### Смесь гауссиан с сохранением моментов

Смесь из `n_components` равновесых гауссиан устроена так, что при любом `radius` её глобальные mean/cov совпадают с $N(\mu, \Sigma)$:

1. работаем в whitened-пространстве, где эталон $\sim N(0, I)$, а затем возвращаемся аффинным преобразованием $X = \mu + LZ$, где $\Sigma = LL^\top$;
2. центры компонент лежат на окружности радиуса `radius` равномерно по углу, поэтому $E[c_i c_i^\top] = \frac{radius^2}{2} I$;
3. дисперсия каждой компоненты $\sigma_c^2 = 1 - radius^2/2$ — вместе с $\frac{radius^2}{2}$ дает ковариацию $I$.

Растущий `radius` раздвигает компоненты, **не меняя глобальных моментов** — это «эксперимент с раздвигающейся смесью» (задачи № 12–13). Ограничения: $radius^2 < 2$ (иначе $\sigma_c^2 \le 0$); `n_components` $= 1$ или $\ge 3$ — при двух компонентах на окружности $E[c\,c^\top] = \mathrm{diag}(r^2, 0)$ и изотропность ломается.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8.5))

panels = [
    (sample_gaussian(1500, seed=0), 'гауссиана N(0, I)'),
    (sample_uniform_cube(1500, seed=0), 'равномерный квадрат [-0.5, 0.5]²'),
    (sample_ball(1500, radius=1.0, seed=0), 'диск (равномерный шар)'),
    (sample_ring(1500, radius=1.0, thickness=0.3, seed=0), 'кольцо'),
    (sample_gaussian_mixture(1500, radius=0.7, n_components=4, seed=0), 'смесь, r=0.7, k=4'),
    (sample_gaussian_mixture(1500, radius=1.4, n_components=4, seed=0), 'смесь, r=1.4, k=4'),
]
for ax, (points, title) in zip(axes.flat, panels):
    ax.scatter(points[:, 0], points[:, 1], s=3, alpha=0.5)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


## 2. Метрики

### 2.1. Класс-обертка `TopologyMetrics`

Реализация — `src/tda_metrics/metrics.py` (импортируется из пакета `tda_metrics`). Задача № 5: передаем два облака точек — получаем метрики через методы, либо все сразу плоским словарем через `compute_all` (одна запись = одна строка `pandas.DataFrame`).

Соглашения:

- **P — «данные» (reference), Q — «модель»**: для асимметричных метрик направление важно; `compute_all` возвращает оба направления MTD (`mtd_PQ`, `mtd_QP`);
- стохастические внутренности библиотек (подсэмплирование в MTD/RTD) фиксируются контекст-менеджером `np_random_seed` — результат не зависит от порядка вызовов;
- MMD / Fréchet / JS реализованы здесь же: несмещенные/стабильные версии с медианной эвристикой и регуляризацией ковариаций (детали в § 2.2).


### 2.2. Детали реализации библиотек

Ответы на вопросы, возникшие при работе с библиотеками (задача № 7).

**MTD: что значат батчи (`batch_size1`, `batch_size2`, `n`)?**

`mtd.mtopdiv(P, Q, batch_size1, batch_size2, n)` строит `n` кросс-баркодов на случайных подвыборках размеров `batch_size1`/`batch_size2` и усредняет скор. Батчи существуют только из-за памяти: матрица попарных расстояний объединения растет как $O((b_1+b_2)^2)$.

Важно: **значения MTD, посчитанные с разными батчами, несравнимы** — сумма длин H1-баров масштабируется с числом точек (демо в § 2.3: на тех же облаках батчи 100/1000 дают 0.5 против 10.9 на полных). Поэтому дефолт обертки — всегда полные облака; `n` (число повторов усреднения) нужен, только если облака больше батчей.

Нюанс: внутри `mtd.calc_cross_barcodes` облака меняются местами («for consistency with the paper»), поэтому параметры батчей меняются ролями; в нашей обертке `batch_size1` ограничивает P, `batch_size2` — Q.

**MTD: почему в старом ноутбуке MTD был ненулевым на «совпадающих» облаках?** *(вопрос из переписки: «облака совпадают полностью, а MTD не ноль»)*

Это не баг метрики: облака совпадали только *визуально* — в старом ноутбуке сравнивались две независимые выборки одного и того же закона (кольца с сидами 42 и 1). На точной копии MTD равен нулю ровно (демо в § 2.3); ненулевое значение на «одинаковых с виду» облаках — это шум сэмплирования, и он нормально, что метрика его видит. Отсюда и совет из переписки использовать разные сиды: тогда уровень шума каждого эксперимента виден на нулевом шаге и калибрует весь график.

**MTD: по каким гомологиям считается расстояние?** *(задача № 8)*

Скор MTD — **сумма длин только H1-баров** кросс-баркода (на баркод-графике библиотеки это зеленые отрезки; синие — H0, «сумма длин синих и зеленых отрезков» из переписки). H1 отвечает за циклы; сумма H0 (связность) доступна отдельно через `mtd_homology()`. Для оценки качества генерации авторы MTopDiv рекомендуют симметризацию: $0.5 \cdot (\mathrm{MTD}(P,Q) + \mathrm{MTD}(Q,P))$.

**RTD: почему `rtd(P, Q) == rtd(Q, P)`?** *(вопрос из старого ноутбука: «тут одинаково — почему?»)*

RTD симметричен по построению: на каждом повторе усредняются оба направления (внутри `rtd1` — баркод объединения, где блок одного облака берется из его собственных нормированных intra-расстояний, а второй — из поэлементного минимума двух; и та же конструкция с ролями P/Q). Внутриоблачные расстояния нормируются на свой 0.9-квантиль, скоп — сумма длин H1-баров. `batch` — размер подвыборки (одни и те же индексы в обоих облаках), `trials` — число повторов усреднения. Ограничение: $|P| = |Q|$.

**Improved precision/recall: что такое `nhood_sizes` и батчи? Зачем был второй вызов?**

`nhood_sizes` — значения k для kNN-оценки многообразия: вокруг каждой точки строится гиперсфера радиуса «расстояние до k-го соседа»; k = 1, 3, 10 — из статьи (меньшие k = более тонкая, но более шумная оценка). `row_batch_size` / `col_batch_size` — чанкинг матрицы попарных расстояний (компромисс память/скорость, на наших размерах не критичен).

Один вызов возвращает **сразу и precision, и recall**: precision — доля точек Q в многообразии P, recall — доля точек P в многообразии Q. Второй вызов с аргументами в обратном порядке (как в старом ноутбуке) возвращает те же числа с точностью до перестановки — он избыточен и убран. TF-граф сбрасывается после каждого вызова (`reset_default_graph`), иначе граф накапливается между вызовами в цикле.

**Наши реализации (симметричные метрики)**

- **MMD** — несмещенная оценка $\widehat{\mathrm{MMD}}^2$ (Gretton et al., 2012) с RBF-ядром и медианной эвристикой $\sigma = median/\sqrt{2}$ по объединенной выборке. В старой версии был фиксированный `gamma=1.0`, т.е. метрика зависела от масштаба данных.
- **Fréchet distance** — формула FID, примененная к облакам как к «фичам»: $d^2 = \|\mu_1-\mu_2\|^2 + \mathrm{tr}(\Sigma_1 + \Sigma_2 - 2(\Sigma_1\Sigma_2)^{1/2})$. Классический FID — та же формула на фичах InceptionV3 (для изображений — torchmetrics); в § 5 применяем её к CLIP-эмбеддингам — это подход CMMD из статьи «Rethinking FID».
- **JS-дивергенция** — непрерывный случай: $JS = H(M) - \tfrac{1}{2}(H(P)+H(Q))$, где $M = 0.5P + 0.5Q$ — смесь (объединенная выборка — честная выборка из $M$); энтропии — kNN-оценка Козаченко–Леоненко; результат в битах, $0 \le JS \le 1$. `scipy.stats.jensenshannon` не подходит: он для дискретных распределений.


### 2.3. Sanity-проверки и базовая задача

1. **Точная копия** (Q = P по точкам): все метрики обязаны принять «идеальные» значения — расстояния ровно 0, precision/recall ровно 1.
2. **Тот же закон, другой сид**: облака *выглядят* одинаково (две выборки одного распределения), но это разные множества точек — метрики дают ненулевой уровень (шум сэмплирования), precision/recall $< 1$. Это разгадка вопроса из переписки: «облака совпадают полностью, а MTD не ноль» — облака совпадали только визуально; корректное поведение, а не баг. Именно поэтому в экспериментах § 3 везде используются разные сиды: уровень шума виден на нулевом шаге и калибрует динамику.
3. **Батчи в MTD**: при батчах меньше облака значение MTD зависит от размера батча (меньше точек — меньше H1-структуры), поэтому для сравнимости между вызовами обертка всегда считает на полных облаках.


In [ ]:
metrics = TopologyMetrics(seed=42, device='cpu')

ring_a = sample_ring(1000, radius=5.0, thickness=1.5, seed=42)
ring_copy = ring_a.copy()
ring_b = sample_ring(1000, radius=5.0, thickness=1.5, seed=1)

display(pd.Series(metrics.compute_all(ring_a, ring_copy), name='точная копия').to_frame().round(4))
display(pd.Series(metrics.compute_all(ring_a, ring_b), name='тот же закон, другой сид').to_frame().round(4))

sub = metrics.mtd(ring_a, ring_b, batch_size1=100, batch_size2=1000, n=1)
full = metrics.mtd(ring_a, ring_b)
print(f'MTD (тот же закон, другой сид): батчи 100/1000 -> {sub:.3f}, полные батчи -> {full:.3f}')
print('Значения MTD при разных батчах несравнимы: сумма H1-баров масштабируется с числом точек.')


In [ ]:
P = sample_gaussian(100, seed=1)
Q = sample_uniform_cube(100, seed=2)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(P[:, 0], P[:, 1], s=18, alpha=0.7, label='P: гауссиана')
ax.scatter(Q[:, 0], Q[:, 1], s=18, alpha=0.7, label='Q: равномерный квадрат')
ax.legend()
ax.set_aspect('equal')
ax.grid(alpha=0.3)
plt.show()

base_task = pd.Series(metrics.compute_all(P, Q), name='гауссиана vs равномерный квадрат')
display(base_task.to_frame().round(4))


## 3. Синтетические эксперименты

Нумерация — из переписки. Общий дизайн:

- **P — «данные» (seed 42), Q — «модель» (seed 7)**: облака всегда семплируются независимо (задача № 9 — никаких копий);
- деформация применяется к модельному облаку (в 3б — к данным), шаг за шагом; противое облако фиксировано;
- на шаге 0 оба облака — выборки одного закона: значения метрик там = уровень шума сэмплирования (§ 2.3), динамику смотрим относительно него;
- в экспериментах со смесью модель пересемплируется на каждом шаге с тем же сидом (метод общих случайных чисел) — траектории гладкие и реагируют только на `radius`.

| № | Эксперимент | Данные (P) | Модель (Q) | Деформация |
|---|---|---|---|---|
| 1 | кольцо → кольцо | кольцо | кольцо | сдвиг по x, 25 шагов |
| 2а | данные — диск | диск | кольцо | сдвиг по x, 25 шагов |
| 2б | данные — кольцо | кольцо | диск | сдвиг по x, 25 шагов |
| 3а | данные — гауссиана | $N(0,I)$ | смесь, $k=4$ | `radius` 0 → 1.4, 15 шагов |
| 3б | данные — смесь | смесь, $k=4$ | $N(0,I)$ | `radius` 0 → 1.4, 15 шагов |
| 4 | число компонент | $N(0,I)$ | смесь, $k$ = 4 / 8 / 16 | `radius` 0 → 1.4, 15 шагов |

**Что здесь интересно:** в 3а/3б смесь раздвигается *с сохранением глобальных моментов* — mean и covariance остаются как у $N(0,I)$ на всех шагах. Значит, Fréchet distance (и классический FID) такой деформации почти не видит, а топологические метрики и precision/recall — видят. Это яркая иллюстрация того, что разные метрики измеряют разное.

**RTD не видит сдвиги.** В экспериментах 1, 2а, 2б траектория RTD — константа (с точностью до шума подсэмплирования). Это свойство, а не баг: матрица для баркода RTD строится только из *внутренних* попарных расстояний каждого облака (без cross-расстояний), поэтому RTD инвариантен к независимым движениям (сдвигам/поворотам) каждого из облаков. Сдвиг он «увидит» только если тот изменит внутреннюю структуру — например, когда часть кольца-model выходит за пределы диска и её плотность меняется. Зато во внутренние деформации (эксперименты 3а/3б/4 — раздвигание смеси) RTD видит отлично. Получается удобная система: сдвиговые эксперименты дискриминируют метрики по инвариантности, эксперименты со смесью — по типу «зрения» (локальная структура vs глобальные моменты).

> **Время выполнения:** весь § 3 — примерно 50–70 минут на T4. Замер на 1000+1000 точек: RTD с 5 усредлениями ≈ 17 c (основная стоимость — 10 запусков ripser++ на вызов `compute_all`), MTD в обе стороны ≈ 3 c, improved PR ≈ 1.2 c, статистические метрики ≈ 0.1 c. Прогресс виден по шагам (tqdm). Точнее/дороже — `compute_all(..., rtd_trials=10)`, быстрее — `rtd_trials=2`.


### 3.1. Эксперимент 1 — кольцо → кольцо (сдвиг)

Два кольца одного закона; модель сдвигается по x на 0.2 за шаг (за 25 шагов — на полный радиус). Ожидаем: рост MTD/MMD/Fréchet/JS и падение precision/recall; на шаге 0 — уровень шума сэмплирования. **RTD остается константой** — инвариантен к сдвигу (см. замечание выше).


In [ ]:
ring_data = sample_ring(N_POINTS, radius=5.0, thickness=1.5, seed=SEED_DATA)
ring_model = sample_ring(N_POINTS, radius=5.0, thickness=1.5, seed=SEED_MODEL)

exp1 = run_experiment(
    'эксп. 1: кольцо vs кольцо, сдвиг',
    data_fn=constant_cloud(ring_data),
    model_fn=translated_cloud(ring_model, eps=0.2),
    n_steps=25,

    metrics_obj=metrics,
)
plot_clouds(constant_cloud(ring_data), translated_cloud(ring_model, eps=0.2), show_steps(25))
plot_trajectories(exp1, 'Эксперимент 1: кольцо (данные) vs кольцо (модель), сдвиг по x')
exp1.round(4)


### 3.2. Эксперимент 2а — данные: диск, модель: кольцо (задача № 11)

Кольцо-модель уезжает из диска-данных. С самого начала recall низкий: центр диска не накрыт кольцом. Precision высокий, пока кольцо внутри диска, и падает, когда оно выходит за край.


In [ ]:
disk_data = sample_ball(N_POINTS, radius=6.0, seed=SEED_DATA)

exp2a = run_experiment(
    'эксп. 2а: диск (данные) vs кольцо (модель), сдвиг',
    data_fn=constant_cloud(disk_data),
    model_fn=translated_cloud(ring_model, eps=0.5),
    n_steps=25,

    metrics_obj=metrics,
)
plot_clouds(constant_cloud(disk_data), translated_cloud(ring_model, eps=0.5), show_steps(25))
plot_trajectories(exp2a, 'Эксперимент 2а: данные — диск, модель — кольцо')
exp2a.round(4)


### 3.3. Эксперимент 2б — данные: кольцо, модель: диск (задача № 11)

Зеркально 2а: диск-модель уезжает из кольца-данных. Теперь наоборот — precision низкий с начала (центр диска-модели не совпадает с кольцом-данными), recall высокий, пока диск накрывает кольцо.


In [ ]:
disk_model = sample_ball(N_POINTS, radius=6.0, seed=SEED_MODEL)

exp2b = run_experiment(
    'эксп. 2б: кольцо (данные) vs диск (модель), сдвиг',
    data_fn=constant_cloud(ring_data),
    model_fn=translated_cloud(disk_model, eps=0.5),
    n_steps=25,

    metrics_obj=metrics,
)
plot_clouds(constant_cloud(ring_data), translated_cloud(disk_model, eps=0.5), show_steps(25))
plot_trajectories(exp2b, 'Эксперимент 2б: данные — кольцо, модель — диск')
exp2b.round(4)


### 3.4. Эксперимент 3а — данные: гауссиана, модель: раздвигающаяся смесь (задача № 12)

Смесь-модель раздвигается от $N(0,I)$ (radius = 0) до почти разделенных компонент (radius = 1.4), глобальные mean/cov не меняются. Ключевой момент: **Fréchet distance почти не видит деформацию** (считает по моментам), а топологические метрики и PR — видят.


In [ ]:
gaussian_data = sample_gaussian(N_POINTS, seed=SEED_DATA)

def mixture_model(step, n_components=4):
    return sample_gaussian_mixture(N_POINTS, radius=0.1 * step, n_components=n_components, seed=SEED_MODEL)

exp3a = run_experiment(
    'эксп. 3а: гауссиана (данные) vs раздвигающаяся смесь (модель)',
    data_fn=constant_cloud(gaussian_data),
    model_fn=mixture_model,
    n_steps=15,

    metrics_obj=metrics,
)
plot_clouds(constant_cloud(gaussian_data), mixture_model, show_steps(15))
plot_trajectories(exp3a, 'Эксперимент 3а: данные — N(0,I), модель — смесь (k=4), radius 0→1.4')
exp3a.round(4)


### 3.5. Эксперимент 3б — данные: раздвигающаяся смесь, модель: гауссиана (задача № 12)

Зеркально 3а: теперь раздвигаются данные, модель-гауссиана фиксирована. Сравнение траекторий 3а и 3б показывает, насколько метрики асимметричны: у симметричных (rtd, mmd, frechet, js) траектории совпадают с точностью до шума, у MTD направления и precision/recall различаются.


In [ ]:
def mixture_data(step, n_components=4):
    return sample_gaussian_mixture(N_POINTS, radius=0.1 * step, n_components=n_components, seed=SEED_DATA)

gaussian_model = sample_gaussian(N_POINTS, seed=SEED_MODEL)

exp3b = run_experiment(
    'эксп. 3б: раздвигающаяся смесь (данные) vs гауссиана (модель)',
    data_fn=mixture_data,
    model_fn=constant_cloud(gaussian_model),
    n_steps=15,

    metrics_obj=metrics,
)
plot_clouds(mixture_data, constant_cloud(gaussian_model), show_steps(15))
plot_trajectories(exp3b, 'Эксперимент 3б: данные — смесь (k=4), модель — N(0,I), radius 0→1.4')
exp3b.round(4)


### 3.6. Эксперимент 4 — число компонент смеси (задача № 13)

Тот же сценарий, что 3а, но $k$ = 4 / 8 / 16. Постановка из переписки: число компонент должно быть параметром («сейчас 4, а можно было бы 8»). Смотрим, как растет «чувствительность» метрик к раздвиганию при большем числе компонент.


In [ ]:
exp4 = pd.concat(
    [
        run_experiment(
            f'эксп. 4: смесь {k} компонент',
            data_fn=constant_cloud(gaussian_data),
            model_fn=lambda step, k=k: mixture_model(step, n_components=k),
            n_steps=15,

            metrics_obj=metrics,
            extra={'n_components': k},
        )
        for k in (4, 8, 16)
    ],
    ignore_index=True,
)

plot_clouds(
    constant_cloud(gaussian_data),
    lambda step: mixture_model(step, n_components=16),
    show_steps(15),
    title='Эксперимент 4 (k=16): раздвигающиеся компоненты',
)

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for metric, ax in zip(['mtd_PQ', 'rtd', 'frechet', 'precision@3'], axes.flat):
    for k, part in exp4.groupby('n_components'):
        ax.plot(part['step'], part[metric], marker='.', markersize=4, label=f'k={k}')
    ax.set_title(metric)
    ax.set_xlabel('шаг')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle('Эксперимент 4: влияние числа компонент смеси')
plt.tight_layout()
plt.show()
exp4.round(4)


In [ ]:
METRIC_GROUPS = {
    'precision/recall': SIMILARITY_METRICS,
    'симметричные': ['rtd', 'mmd', 'frechet', 'js'],
    'асимм. топологические': ['mtd_PQ', 'mtd_QP'],
}

ALL_METRICS = [m for metrics in METRIC_GROUPS.values() for m in metrics]


def trajectories_correlation(df):
    """Средняя по экспериментам корреляция Спирмена между траекториями метрик.

    PR-метрики инвертируются (1 - x); константные траектории отбрасываются.
    """
    inverted = df.copy()
    pr_cols = [c for c in inverted.columns if c.startswith(('precision', 'recall'))]
    inverted[pr_cols] = 1.0 - inverted[pr_cols]

    matrices = []
    for _, part in inverted.groupby('experiment'):
        usable = [m for m in ALL_METRICS if m in part.columns and part[m].nunique() > 1]
        if len(part) < 4 or len(usable) < 2:
            continue
        corr = part[usable].corr(method='spearman').reindex(index=ALL_METRICS, columns=ALL_METRICS)
        matrices.append(corr.values)
    averaged = np.nanmean(np.stack(matrices), axis=0)
    return pd.DataFrame(averaged, index=ALL_METRICS, columns=ALL_METRICS)


def plot_correlation_heatmap(corr, title=''):
    fig, ax = plt.subplots(figsize=(11, 9))
    image = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr)), labels=corr.columns, rotation=90)
    ax.set_yticks(range(len(corr)), labels=corr.index)
    for i in range(len(corr)):
        for j in range(len(corr)):
            if not np.isnan(corr.values[i, j]):
                ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=8)
    fig.colorbar(image, ax=ax, shrink=0.8)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def group_correlation_summary(corr):
    """Средняя корреляция внутри групп и между группами (проверка гипотезы)."""
    rows = []
    for group_a, metrics_a in METRIC_GROUPS.items():
        for group_b, metrics_b in METRIC_GROUPS.items():
            values = [
                corr.loc[a, b]
                for a in metrics_a
                for b in metrics_b
                if a != b and not np.isnan(corr.loc[a, b])
            ]
            rows.append({
                'группа A': group_a,
                'группа B': group_b,
                'средняя корреляция': float(np.mean(values)),
            })
    return pd.DataFrame(rows).pivot(index='группа A', columns='группа B', values='средняя корреляция')


In [ ]:
all_experiments = pd.concat([exp1, exp2a, exp2b, exp3a, exp3b, exp4], ignore_index=True)

corr = trajectories_correlation(all_experiments)
plot_correlation_heatmap(corr, 'Корреляции Спирмена между метриками (средние по экспериментам § 3)')
display(group_correlation_summary(corr).round(3))
corr.round(3)


## 5. Реальные данные: CLIP-эмбеддинги

Синтетика управляемая, но «игрушечная». Здесь тот же набор метрик применяется к распределениям **эмбеддингов реальных данных**: картинки проходят через предобученный CLIP ViT-B/32, облако 512-мерных L2-нормализованных векторов — «выборка из распределения».

Выбор CLIP не случаен: в статье «Rethinking FID» (из списка литературы проекта) метрика CMMD — MMD на CLIP-эмбеддингах — предложена как замена FID. Связка «CLIP + `compute_all`» покрывает CMMD и добавляет топологию.

Сценарии — прямые аналоги синтетики:

| Реальный сценарий | Синтетический аналог |
|---|---|
| MNIST train vs MNIST test | тот же закон, разные сиды (§ 2.3) |
| MNIST vs Fashion-MNIST | далекие распределения |
| CIFAR-10 → CIFAR-10-C, severity 1→5 | сдвиг кольца (§ 3.1) |
| модель = цифры 0..k, k = 1..9 | раздвигающаяся смесь (§ 3.4) |

> **Оговорки:** в 512 измерениях kNN-оценки (JS, precision/recall) менее надежны, чем в 2D (концентрация расстояний); абсолютные значения метрик между разными пространствами признаков несравнимы. Предмет анализа — поведение метрик относительно друг друга. Время раздела: ~15 минут (из них ~4 — извлечение эмбеддингов).


In [ ]:
!pip install -q ftfy regex
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q imagecorruptions


In [ ]:
import torch
import clip

from tda_metrics.embeddings import to_rgb, clip_embeddings, get_embeddings, subsample

device = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model, preprocess = clip.load('ViT-B/32', device=device)
print('CLIP ViT-B/32 на', device)

N_EMB = 1500


### 5.1. Датасеты и эмбеддинги

MNIST (train/test), Fashion-MNIST и CIFAR-10 (test) — скачиваются torchvision, эмбеддинги кэшируются в `./embeddings` (повторный запуск ячейки не пересчитывает). Извлечение ~90k картинок ≈ 3–4 минуты на T4.


In [ ]:
from torchvision import datasets as tvd

mnist_train = tvd.MNIST('data', train=True, download=True)
mnist_test = tvd.MNIST('data', train=False, download=True)
fashion_test = tvd.FashionMNIST('data', train=False, download=True)
cifar10_test = tvd.CIFAR10('data', train=False, download=True)

emb_mnist_train = get_embeddings('mnist_train', clip_model, preprocess, device, mnist_train.data.numpy())
emb_mnist_test = get_embeddings('mnist_test', clip_model, preprocess, device, mnist_test.data.numpy())
emb_fashion_test = get_embeddings('fashion_test', clip_model, preprocess, device, fashion_test.data.numpy())
emb_cifar10 = get_embeddings('cifar10_test', clip_model, preprocess, device, cifar10_test.data)

for name, X in [('mnist_train', emb_mnist_train), ('mnist_test', emb_mnist_test),
                ('fashion_test', emb_fashion_test), ('cifar10_test', emb_cifar10)]:
    print(f'{name}: {X.shape}')


### 5.2. Sanity и далекие распределения

MNIST train vs test — «тот же закон, разные выборки» (аналог § 2.3): метрики должны быть малыми. MNIST vs Fashion-MNIST — далекие распределения: метрики большие. Отношение этих двух чисел — грубая «разрешающая способность» каждой метрики на реальных данных.


In [ ]:
rows = {
    'MNIST train vs test (sanity)': metrics.compute_all(
        subsample(emb_mnist_train, N_EMB, seed=SEED_DATA),
        subsample(emb_mnist_test, N_EMB, seed=SEED_MODEL),
    ),
    'MNIST vs Fashion-MNIST': metrics.compute_all(
        subsample(emb_mnist_test, N_EMB, seed=SEED_DATA),
        subsample(emb_fashion_test, N_EMB, seed=SEED_MODEL),
    ),
}
display(pd.DataFrame(rows).round(4))


### 5.3. CIFAR-10 → CIFAR-10-C: постепенная деградация

Прямой аналог сдвиговых экспериментов: фиксированный набор из 4000 картинок CIFAR-10, к каждому применяется `gaussian_noise` нарастающей силы (severity 1→5 — те же типы порч, что в бенчмарке CIFAR-10-C). Данные — чистые эмбеддинги, модель — испорченные; severity = шаг.


In [ ]:
from imagecorruptions import corrupt

corruption = 'gaussian_noise'
severities = [1, 2, 3, 4, 5]
base_idx = np.random.default_rng(SEED_DATA).choice(len(cifar10_test.data), 4000, replace=False)
base_images = list(cifar10_test.data[base_idx])

clean = get_embeddings('cifar_clean', clip_model, preprocess, device, base_images)
corrupted = {
    severity: get_embeddings(
        f'cifar_{corruption}_s{severity}',
        [corrupt(img, corruption_name=corruption, severity=severity) for img in base_images],
    )
    for severity in severities
}

df_cifar = run_experiment(
    f'CIFAR-10-C: {corruption}, severity 1->5',
    data_fn=lambda step: subsample(clean, N_EMB, seed=SEED_DATA),
    model_fn=lambda step: subsample(corrupted[severities[step]], N_EMB, seed=SEED_MODEL),
    n_steps=len(severities),

    metrics_obj=metrics,
)
plot_trajectories(df_cifar, 'CIFAR-10-C: гауссовский шум нарастающей силы (severity 1→5)')
df_cifar.round(4)


### 5.4. MNIST: рост числа классов в модели

Аналог раздвигающейся смеси: модель видит только цифры $0..k$ ($k$ = 1..9), данные — все 10 классов. С ростом $k$ модель приближается к данным — расходимости падают, precision/recall растут. $k = 1$ — одна «компонента», $k = 9$ — почти полное покрытие.


In [ ]:
targets = mnist_train.targets.numpy()


def class_embeddings(c, n_per_class, seed):
    idx = np.where(targets == c)[0]
    selected = np.random.default_rng(seed).choice(idx, size=n_per_class, replace=False)
    return emb_mnist_train[selected]


per_class = {c: class_embeddings(c, n_per_class=N_EMB, seed=SEED_DATA + c) for c in range(10)}

df_mnist_classes = run_experiment(
    'MNIST: рост числа классов в модели',
    data_fn=lambda step: subsample(emb_mnist_train, N_EMB, seed=SEED_DATA),
    model_fn=lambda step: subsample(
        np.concatenate([per_class[c] for c in range(step + 1)]),
        N_EMB,
        seed=SEED_MODEL,
    ),
    n_steps=9,

    metrics_obj=metrics,
)
plot_trajectories(df_mnist_classes, 'MNIST: модель = цифры 0..k, k = 1..9; данные — все цифры')
df_mnist_classes.round(4)


### 5.5. Корреляции метрик на реальных данных

Тот же анализ, что в § 4, но на траекториях § 5 (CIFAR-10-C + рост классов). Сравниваем с синтетикой: держится ли группировка «precision/recall ↔ симметричные ↔ асимметричные топологические» на реальных распределениях.


In [ ]:
df_real = pd.concat([df_cifar, df_mnist_classes], ignore_index=True)

corr_real = trajectories_correlation(df_real)
plot_correlation_heatmap(corr_real, 'Корреляции метрик: реальные данные (CLIP)')
display(group_correlation_summary(corr_real).round(3))
corr_real.round(3)


### 5.6. (Опционально) Сырые пиксели vs CLIP

Тот же вопрос «MNIST vs Fashion-MNIST», но «эмбеддинги» — сами пиксели (784-dim). Абсолютные значения между строками **несравнимы** (разные пространства и масштабы) — смотрим на паттерн: насколько каждая метрика видит разницу распределений в исходном пространстве vs пространстве предобученной модели. Это проверка чувствительности выводов к выбору признакового пространства.


In [ ]:
pixels_mnist = mnist_train.data.numpy().reshape(len(mnist_train), -1).astype(np.float32) / 255.0
pixels_fashion = fashion_test.data.numpy().reshape(len(fashion_test), -1).astype(np.float32) / 255.0

rows = {
    'CLIP: MNIST vs Fashion': metrics.compute_all(
        subsample(emb_mnist_test, N_EMB, seed=SEED_DATA),
        subsample(emb_fashion_test, N_EMB, seed=SEED_MODEL),
    ),
    'пиксели: MNIST vs Fashion': metrics.compute_all(
        subsample(pixels_mnist, N_EMB, seed=SEED_DATA),
        subsample(pixels_fashion, N_EMB, seed=SEED_MODEL),
    ),
}
display(pd.DataFrame(rows).round(4))


## 6. Итоги

**Инженерное:** единый пайплайн «данные → `TopologyMetrics.compute_all` → `DataFrame` → графики/корреляции»; все задачи из переписки закрыты (таблица в начале ноутбука); библиотечные грабли задокументированы в § 2.2 (батчи MTD, симметричность RTD, nhood_sizes, TF1-режим).

**Что показали эксперименты (свойства метрик, а не только цифры):**

1. **Корректность**: на точной копии все метрики дают ровно 0/1 (§ 2.3); ненулевые значения на «одинаковых с виду» облаках — шум сэмплирования двух независимых выборок, он же калибрует нулевой шаг каждого эксперимента.
2. **RTD инвариантен к сдвигам** (§ 3.1–3.3): траектория — константа, потому что матрица RTD строится только из внутренних попарных расстояний облаков. Сдвиговые эксперименты его не измеряют — измеряют внутренние деформации (§ 3.4–3.6).
3. **Fréchet distance (и классический FID) слеп к раздвиганию смеси с сохранением моментов** (§ 3.4–3.5): ≈ 0.1 на всех 15 шагах, пока JS растет до 0.75, MMD и топологические метрики реагируют. Метрики «по моментам» не видят мультимодальность такого типа.
4. **Precision/recall-картина mode dropping** (§ 3.4–3.5): при разделении смеси precision@3 и recall@3 расходятся зеркально (0.98/0.18 в одном направлении, 0.18/0.99 в другом) — ровно та семантика, ради которой PR-метрики и существуют.
5. **Асимметрия MTD** (§ 3.4–3.5): направления $P \to Q$ и $Q \to P$ отвечают на одну деформацию по-разному (7.8→11.4 и 7.6→4.4 на тех же данных).
6. **Корреляционный анализ** (§ 4): рабочая гипотеза «PR-метрики похожи между собой сильнее, чем с симметричными» в чистом виде **не подтвердилась**: внутри-PR 0.34 против PR↔симметричные 0.34. Реальная структура интереснее — метрики группируются по *измеряемому аспекту*, а не по семейству: `recall@k` + `mmd` + `js` образуют кластер «покрытия» (0.6–0.8); в экспериментах со смесью precision и recall расходятся в противоположные стороны (mode dropping: 0.98/0.18), и так же расходятся направления MTD (в итоге `mtd_PQ`↔`mtd_QP` ≈ 0); `rtd` антикоррелирует с `mmd`/`js` (−0.7): при раздвигании смеси расходимость растет, а внутренняя структура облаков упрощается — rtd падает. На реальных данных (§ 5) картина похожая: монотонные траектории дают идеальную корреляцию mmd/frechet/js, rtd снова «сам по себе» (−0.9), направления MTD совпадают (0.58). Оговорка: в § 5 всего два эксперимента, корреляции там грубые.
7. **Реальные данные** (§ 5): тот же аппарат на CLIP-эмбеддингах (подход CMMD из «Rethinking FID»). Паттерны повторяются: рост классов в модели поднимает recall (0.12 → 0.94) при стабильном precision (~0.83) — подмножество классов это «модель без фальшивых мод, но с неполным покрытием»; CIFAR-10-C дает монотонную деградацию по mmd/Fréchet/precision. Важная оговорка: **MTD/RTD на 512-мерных CLIP-эмбеддингах при N=1500 дискриминируют слабо** (mtd меняется на десятки процентов, тогда как mmd — на порядок) — концентрация расстояний в высоких размерностях ослабляет топологические сигналы; для топологических метрик на эмбеддингах нужны б'ольшие выборки, и выводы по ним на реальных данных надо делать осторожно.

**Что дальше (естественные продолжения):**

- обучить простую генеративную модель (VAE/диффузия на MNIST) и сравнивать реал/генерации по чекпоинтам — «динамика обучения» глазами всех метрик;
- другие типы порч CIFAR-10-C (`pixelate`, `gaussian_blur`, ...);
- другие бэкбоны (ResNet-50, DINOv2) — устойчивость корреляционной картины к выбору признакового пространства;
- формальные статистические тесты для корреляций (перестановочные), доверительные интервалы метрик через бутстрап облаков.


## Литература

**Метрики:**

- Barannikov S., Trofimov I., Sotnikov G., et al. *Manifold Topology Divergence: a Framework for Comparing Data Manifolds.* NeurIPS 2021. [arXiv:2106.04024](https://arxiv.org/abs/2106.04024) — MTD; библиотека [MTopDiv](https://github.com/IlyaTrofimov/MTopDiv).
- Barannikov S., Trofimov I., Balabin N., Burnaev E. *Representation Topology Divergence: A Method for Comparing Neural Network Representations.* ICML 2022. [arXiv:2201.00058](https://arxiv.org/abs/2201.00058) — RTD; библиотека [RTD](https://github.com/IlyaTrofimov/RTD).
- Kynkäänniemi T., Karras T., Laine S., Lehtinen J., Aila T. *Improved Precision and Recall for Assessing Generative Models.* NeurIPS 2019. [arXiv:1904.06991](https://arxiv.org/abs/1904.06991) — improved precision/recall; [форк для Colab](https://github.com/AlimAlb/improved-precision-and-recall-metric).
- Heusel M., Ramsauer H., Unterthiner T., Nessler B., Hochreiter S. *GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium.* NeurIPS 2017. [arXiv:1706.08500](https://arxiv.org/abs/1706.08500) — Fréchet Inception Distance.
- Gretton A., Borgwardt K., Rasch M., Schölkopf B., Smola A. *A Kernel Two-Sample Test.* JMLR 2012 — MMD.
- Kozachenko L., Leonenko N. *Sample Estimate of the Entropy of a Random Vector.* Probl. Inf. Transm. 1987 — kNN-оценка энтропии (используется в JS).
- Lin J. *Divergence measures based on the Shannon entropy.* IEEE Trans. Inf. Theory 1991 — дивергенция Дженсена–Шеннона.

**Из списка научного руководителя:**

- Jayasumana S. et al. *Rethinking FID: Towards a Better Evaluation Metric for Image Generation.* 2024. [arXiv:2401.09603](https://arxiv.org/abs/2401.09603) — CMMD (CLIP + MMD), критика FID.
- *Manifoldron: Direct Space Partition via Manifold Discovery.* 2022. [arXiv:2201.05279](https://arxiv.org/abs/2201.05279).
- *Topological Alternatives for Precision and Recall in Generative Models.* IEEE 2025. [ieeexplore.ieee.org/document/11123796](https://ieeexplore.ieee.org/abstract/document/11123796).

**Инструменты:**

- Zhang S. et al. *Ripser++: A GPU-based tool for computing persistence barcodes.* [github.com/simonzhang00/ripser-plusplus](https://github.com/simonzhang00/ripser-plusplus) — персистентные гомологии на GPU.
- Radford A. et al. *Learning Transferable Visual Models From Natural Language Supervision.* ICML 2021 — CLIP.
- Hendrycks D., Dietterich T. *Benchmarking Neural Network Robustness to Common Corruptions and Perturbations.* ICLR 2019 — CIFAR-10-C, пакет `imagecorruptions`.
